# Amaliyot 3. Lemmatizatsiya va stemming

3-interaktiv laboratoriyaga («Morfologiya va lemmatizatsiya») hamroh. Daftar oʻz-oʻzidan yetarli: birinchi katak `nlp_common.py` faylini yaratadi (oʻzbek tilining oʻquv korpusi: morfologik belgilangan shablon gaplar va yordamchi funksiyalar). Kataklarni yuqoridan pastga ishga tushiring. Haqiqiy masalada oʻquv korpusni oʻzingiznikiga almashtiring.

In [ ]:
%%writefile nlp_common.py
"""Oʻquv moduli: oʻzbek tili uchun korpus generatorlari (agglyutinatsiya qoidalari bilan)."""
import random, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110; plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = .25
LANG = "uz"
VOWELS = "aeiouʻ"
def is_vowel_final(w):
    return w[-1] in "aeiou" or w.endswith("oʻ")
POSS_V = {"1sg": "m", "2sg": "ng", "3sg": "si", "1pl": "miz", "2pl": "ngiz"}
POSS_C = {"1sg": "im", "2sg": "ing", "3sg": "i", "1pl": "imiz", "2pl": "ingiz"}
CASE = {"nom": "", "gen": "ning", "acc": "ni", "dat": "ga", "loc": "da", "abl": "dan"}
ALT = {"qishloq": "qishlogʻ", "koʻylak": "koʻylagʻ"}   # oxirgi undosh almashinuvi (egalik oldidan)

def attach(root, plural=False, poss=None, case="nom"):
    """Ildizga ketma-ket qoʻshimchalar: ildiz + (-lar) + egalik + kelishik."""
    w = ALT.get(root, root) if (poss and not plural and root in ALT) else root
    if plural: w += "lar"
    if poss: w += (POSS_V if is_vowel_final(w) else POSS_C)[poss]
    return w + CASE[case]

SUBJ = ["ona","ota","bola","talaba","oʻqituvchi"]; OBJ = ["kitob","daftar","gazeta","xat","qalam"]; PLACE = ["kutubxona","maktab","xona","uy"]
ADJ = ["yangi","qiziqarli","yaxshi","eski"]; VERB = {"oʻqimoq":"oʻqiydi","yozmoq":"yozadi","koʻrmoq":"koʻradi","sevmoq":"sevadi"}

def make_sentences(n=500, seed=0):
    """Gaplar morfologik belgilash bilan: tokens, tags (UPOS), lemmas (SOV tartibi)."""
    r = random.Random(seed); out = []
    for _ in range(n):
        t, g, l = [], [], []
        if r.random() < .4: a = r.choice(ADJ); t.append(a); g.append("ADJ"); l.append(a)
        s = r.choice(SUBJ); t.append(attach(s, r.random() < .25, r.choice([None, None, "1sg"]))); g.append("NOUN"); l.append(s)
        if r.random() < .4: p = r.choice(PLACE); t.append(attach(p, False, r.choice([None, "1sg"]), "loc")); g.append("NOUN"); l.append(p)
        if r.random() < .4: a = r.choice(ADJ); t.append(a); g.append("ADJ"); l.append(a)
        o = r.choice(OBJ); t.append(attach(o, r.random() < .25, r.choice([None, "1sg", "3sg"]), r.choice(["nom", "acc"]))); g.append("NOUN"); l.append(o)
        v = r.choice(list(VERB)); t.append(VERB[v]); g.append("VERB"); l.append(v)
        out.append(dict(tokens=t, tags=g, lemmas=l))
    return out

def all_forms():
    """Barcha shakllar eталон lemma bilan: [(shakl, lemma)]  (alternatsiyali ildizlar ham)."""
    roots = SUBJ + OBJ + PLACE + list(ALT)
    return sorted({(attach(r, p, o, c), r) for r in roots for p in (False, True) for o in (None, *POSS_V) for c in CASE})

_A = ["mushuk","it","ot","sigir"]; _F = ["baliq","goʻsht","non","pishloq"]; _D = ["sut","suv","choy","sharbat"]; _V = ["mashina","avtobus","poyezd","tramvay"]
def make_topic_sentences(n=2000, seed=0):
    r = random.Random(seed); fr = [lambda: [r.choice(_A), r.choice(_F), "yeydi"], lambda: [r.choice(_A), r.choice(_D), "ichadi"], lambda: [r.choice(_A), "uyda", "uxlaydi"],
        lambda: [r.choice(_V), "yoʻlda", "ketadi"], lambda: [r.choice(_V), "bekatda", "turadi"]]
    return [r.choice(fr)() for _ in range(n)]
CATS = {w: c for c, ws in enumerate([_A, _F + _D, _V]) for w in ws}

def make_reviews(n=400, seed=0):
    r = random.Random(seed); N = ["film","mahsulot","xizmat","ovqat","ish"]
    P = ["ajoyib","yaxshi","zoʻr","mazali"]; B = ["yomon","dahshatli","zerikarli","sifatsiz"]; rows = []
    for _ in range(n):
        pos = r.random() < .5; adj = r.choice(P if pos else B); neg = r.random() < .3
        pat = r.choice(["{n} {a}", "{n} juda {a}"]) if not neg else "{n} {a} emas"
        rows.append((pat.format(n=r.choice(N), a=adj), int(pos != neg)))
    return pd.DataFrame(rows, columns=["text", "label"])

_PM = ["Ali","Vali","Dilshod","Akmal","Bobur"]; _PF = ["Madina","Dilnoza","Zarina","Gulnora"]
_SM = ["Karimov","Valiyev","Rahimov","Ahmedov"]; _SF = [s + "a" for s in _SM]; _C = ["Toshkent","Samarqand","Buxoro","Moskva","Qozon"]
def make_ner(n=500, seed=0):
    """[(tokens, BIO-teglar)]"""
    r = random.Random(seed); out = []
    for _ in range(n):
        f = r.random() < .5; c = r.choice(_C); nm = r.choice(_PF if f else _PM); sn = r.choice(_SF if f else _SM); k = r.random()
        if k < .4: t = [nm, sn, attach(c, case="dat"), "keldi"]; g = ["B-PER","I-PER","B-LOC","O"]
        elif k < .8: t = [nm, sn, attach(c, case="loc"), "yashaydi"]; g = ["B-PER","I-PER","B-LOC","O"]
        else: t = [nm, attach(c, case="abl"), "keldi"]; g = ["B-PER","B-LOC","O"]
        out.append((t, g))
    return out

_PL = ["vokzal","aeroport","markaz","universitet","bozor"]
def make_intents(n=400, seed=0):
    r = random.Random(seed); rows = []
    T = {"book": ["{f}dan {t}ga taksi kerak","{t}ga taksi buyurtma qilmoqchiman","{f}dan taksi chaqiring","menga taksi kerak","taksi chaqirmoqchiman","{t}ga bormoqchiman"],
         "cancel": ["buyurtmani bekor qiling","buyurtma bekor","taksini bekor qilmoqchiman"], "greet": ["salom","assalomu alaykum","xayrli kun"], "thanks": ["rahmat","katta rahmat","rahmat sizga"]}
    for _ in range(n):
        k = r.choice(list(T)); rows.append((r.choice(T[k]).format(f=r.choice(_PL), t=r.choice(_PL)), k))
    return pd.DataFrame(rows, columns=["text", "intent"])

def make_parallel(n=200, seed=0):
    r = random.Random(seed); S = {"ona":"mother","ota":"father","qiz":"girl","talaba":"student"}; V = {"oʻqiydi":"reads","yozadi":"writes","koʻradi":"sees"}
    O = {"kitob":"book","gazeta":"newspaper","xat":"letter","rasm":"picture"}; out = []
    for _ in range(n):
        s, v, o = r.choice(list(S)), r.choice(list(V)), r.choice(list(O)); out.append(([s, o, v], [S[s], V[v], O[o]]))
    return out

ARTICLE = ["Toshkentda zamonaviy rangtasvir boʻyicha yangi koʻrgazma ochildi.","Koʻrgazma turli mamlakatlardan kelgan oʻttiz nafar rassom ishlarini birlashtirdi.",
 "Tashkilotchilar zamonaviy rangtasvir koʻrgazmasi koʻp tashrif buyuruvchilarni jalb qilganini taʼkidlashdi.","Rassomlar shahar va uning aholisiga bagʻishlangan rasmlarni taqdim etdi.",
 "Tashrif buyuruvchilar ayniqsa yosh rassomlarning ishlarini alohida taʼkidladi.","Koʻrgazma oy oxirigacha davom etadi, talabalar uchun kirish bepul.","Shaharda bu kunlarda ob-havo iliq va quyoshli boʻlib turibdi."]
PASSAGES = ["Toshkent Oʻzbekistonning poytaxti va mamlakatdagi eng yirik shahar boʻlib, aholisi ikki milliondan ortiq.","Toshkent metrosi 1977-yilda ochilgan va Markaziy Osiyodagi birinchi metropoliten boʻlgan.",
 "Sirdaryo Oʻzbekiston, Qozogʻiston va Tojikiston hududidan oqib oʻtadi.","Samarqand qadimiy masjidlari va Registon maydoni bilan mashhur.","Oʻzbek oshxonasi palovi bilan mashhur, uni katta qozonda pishirishadi."]
QUESTIONS = [("Oʻzbekistonning yirik shahri Toshkentda yer osti transporti qachon paydo boʻlgan?", 1, "1977-yilda"), ("Oʻzbekistonda qaysi taom eng sevimli hisoblanadi?", 4, "palov"), ("Qaysi suv yoʻli bir nechta davlat orqali oqadi?", 2, "Sirdaryo")]
AMBIG = [("Men koʻk oldim.", ["koʻkat","koʻk rangli narsa"]), ("Men durbin bilan odamni koʻrdim.", ["men durbin orqali koʻrdim","odamda durbin bor edi"]), ("Yoz.", ["yozmoq feʼlining buyruq shakli","fasl"])]

In [ ]:
from nlp_common import *

## 1. Базовый стеммер: слои суффиксов

In [ ]:
CASES = ["ning", "dagi", "gacha", "ni", "ga", "da", "dan"]; POSS = ["imiz", "ingiz", "im", "ing", "i"]; PLUR = ["lar"]
def strip_layers(w, layers):
    for L in layers:
        c = next((s for s in sorted(L, key=len, reverse=True) if w.endswith(s) and len(w) - len(s) >= 2), None)
        if c: w = w[:-len(c)]
    return w
forms = all_forms()                                # [(shakl, etalon lemma)]
def evaluate(fn):
    return np.mean([fn(f) == l for f, l in forms])
base = lambda w: strip_layers(w, [CASES, POSS, PLUR])
print("Bazaviy stemmer aniqligi:", round(evaluate(base), 3))
bad = [(f, base(f), l) for f, l in forms if base(f) != l]
print("Xato misollari:", bad[:6])

## 2. Qoidalarni bosqichma-bosqich yaxshilash

In [ ]:
POSS_V = ["miz", "ngiz", "si", "ng", "m"]           # unlidan keyin egalik qoʻshimchalari
ALT_INV = {"qishlogʻ": "qishloq", "koʻylagʻ": "koʻylak"}   # oxirgi undosh almashinuvi
def better(w):
    r = strip_layers(w, [CASES])
    for s in sorted(POSS + POSS_V, key=len, reverse=True):
        if r.endswith(s) and len(r) - len(s) >= 2:
            cand = r[:-len(s)]
            if s in POSS_V and cand[-1] not in "aeiou": continue      # unli ildizdan keyin kelgan shakl
            r = cand; break
    r = strip_layers(r, [PLUR])
    return ALT_INV.get(r, r)
print("Yaxshilangan stemmer aniqligi:", round(evaluate(better), 3))
print("Hali xato:", [(f, better(f), l) for f, l in forms if better(f) != l][:6])

## Topshiriqlar

1. Qaysi shakllarda yaxshilangan stemmer hali ham xato qiladi? Nima uchun?
2. `ALT_INV` ni lugʻatsiz, qoida bilan almashtirib koʻring (masalan, «gʻ» → «q» oxirgi harf sifatida).
3. Aniqlikni qoʻshimcha zanjir uzunligi (0, 1, 2, 3 ta qoʻshimcha) boʻyicha hisoblang. Qaysi uzunlikda xatolar koʻpayadi?